In [1]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 


In [ ]:
def print_json_structure(obj, indent=0):
    pad = "  " * indent

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{pad}{key}: {type(value).__name__}")
            print_json_structure(value, indent + 1)

    elif isinstance(obj, list):
        print(f"{pad}[list] len={len(obj)}")
        if obj:
            print_json_structure(obj[0], indent + 1)

TO DO:

- get list of air quality sensors (Golemio)
- get list of microclimate sensors (Golemio)
- get list of Weather stations (CHMI)

- join this to one dataset mapping the relevent locations together

- download relevant datasources, joint them based on mapping defined above 

- do analysis (...)

In [ ]:
#data chmi

#https://opendata.chmi.cz/meteorology/climate/historical/data/1hour/



In [ ]:
def get_chmi_stations_metadata():
    url = 'https://opendata.chmi.cz/'
    route = '/meteorology/climate/historical/metadata/meta1.json'

    headers = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)
    
    data = response.json()
    data = data['data']['data']

    headers = data['header'].split(',')
    values = data['values']

    df = pd.DataFrame(values, columns=headers)
    df.to_csv("chmi_stations_metadata.csv", index=False, encoding="utf-8-sig")

get_chmi_stations_metadata()


In [24]:
chmi_url = 'https://opendata.chmi.cz/'
route = '/meteorology/climate/historical/metadata/meta1.json'

headers = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

response = requests.get(f'{chmi_url}{route}', headers = headers, timeout = 60)

chmi_stations = response.json()

chmi_stations = chmi_stations['data']['data']


In [25]:
headers = chmi_stations['header'].split(',')
values = chmi_stations['values']

df_chmi_stat = pd.DataFrame(values, columns=headers)



In [26]:
### dtaa processing

df_chmi_stat = df_chmi_stat[df_chmi_stat['FULL_NAME'].str.contains('Praha', case=False, na=False)]

df_chmi_stat["END_DATE_DT"] = pd.to_datetime(df_chmi_stat["END_DATE"], utc=True, errors="coerce")

df_chmi_stat = (
    df_chmi_stat.sort_values("END_DATE_DT")
    .drop_duplicates(subset="WSI", keep="last")
)

now_utc = pd.Timestamp.now(tz="UTC")
df_chmi_stat = df_chmi_stat[
    (df_chmi_stat["END_DATE_DT"] >= now_utc)
]

In [27]:
df_chmi_stat[df_chmi_stat['FULL_NAME']=='Praha, Klementinum']

,WSI,GH_ID,BEGIN_DATE,END_DATE,FULL_NAME,GEOGR1,GEOGR2,ELEVATION,END_DATE_DT
2217,0-203-0-11515,P1PKLM01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416436,50.086341,190.7,3999-12-31 23:59:00+00:00
2214,0-203-0-11514,P1PKLE01,2023-01-01T00:00:00Z,3999-12-31T23:59:00Z,"Praha, Klementinum",14.416923,50.086634,190.7,3999-12-31 23:59:00+00:00


In [28]:
### discionary of wsi codes

wsi_dict = dict(
    zip(
        df_chmi_stat["WSI"].astype(str),
        df_chmi_stat["FULL_NAME"].astype(str)  
    )
)

In [ ]:
### chmi data variables


In [ ]:
def get_chmi_variables_metadata():
    url = 'https://opendata.chmi.cz/'
    route = '/meteorology/climate/historical/metadata/meta2.json'

    headers = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)

    response = response.json()

    data = response['data']['data']

    headers = data['header'].split(',')
    values = data['values']

    df = pd.DataFrame(values, columns=headers)
    df.to_csv("chmi_variables_metadata.csv", index=False, encoding="utf-8-sig")

get_chmi_variables_metadata()

In [ ]:
chmi_url = 'https://opendata.chmi.cz/'
route = '/meteorology/climate/historical/metadata/meta2.json'

headers = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

response = requests.get(f'{chmi_url}{route}', headers = headers, timeout = 60)

chmi_vars = response.json()

chmi_vars = chmi_vars['data']['data']
headers = chmi_vars['header'].split(',')
values = chmi_vars['values']

df_chmi_vars = pd.DataFrame(values, columns=headers)



In [ ]:
### filtering only needed ones

df_chmi_vars = df_chmi_vars[df_chmi_vars['WSI'].astype(str).isin(wsi_dict)]

In [157]:
df_chmi_vars[df_chmi_vars['EG_EL_ABBREVIATION'] == 'E']

,OBS_TYPE,WSI,BEGIN_DATE,END_DATE,EG_EL_ABBREVIATION,NAME,UN_DESCRIPTION,HEIGHT,SCHEDULE
58414,HLY,0-20000-0-11519,2022-09-01T00:00:00Z,3999-12-31T23:59:00Z,E,Tlak páry,hPa,1.99,1H


In [ ]:
### dictionary of variables abbreviations and names
chmi_vars_dict = dict(
    zip(
        df_chmi_vars['EG_EL_ABBREVIATION'].astype(str),
        df_chmi_vars['NAME'].astype(str)  
    )
)

In [ ]:
### golemio data 


##### air quality stations 

## metadata: https://opendata.chmi.cz//air_quality/recent/metadata/metadata.json 

In [17]:
def set_api_key(token = 'api_key.env'):
    load_dotenv(token)
    api_key = os.getenv('GOLEMIO_API_KEY')
    if api_key:
        print("✓ API key loaded")
    else:
        print("✗ API key not found")
    return api_key

api_key = set_api_key()


✓ API key loaded


In [20]:
def get_airquality_stations_metadata():
    url = 'https://api.golemio.cz/'
    route = '/v2/airqualitystations'

    headers = {
        'X-access-token': api_key, 
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    response = requests.get(f'{url}{route}', headers = headers, timeout = 60)

    data = response.json()
    df = pd.json_normalize(data['features'])

    tmp = df.explode("properties.measurement.components", ignore_index=True)
    dirs = pd.json_normalize(
        tmp["properties.measurement.components"]
    ).add_prefix("properties.measurement.components.")

    df = pd.concat([tmp.drop(columns=["properties.measurement.components"]), dirs], axis=1)

    df.to_csv("airquality_stations_metadata.csv", index=False, encoding="utf-8-sig")

get_airquality_stations_metadata()

In [205]:
api_url = 'https://api.golemio.cz/'
route = '/v2/airqualitystations'

headers = {
    'X-access-token': api_key, 
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

params = {

}

response = requests.get(f'{api_url}{route}', headers = headers, timeout = 60)

data = response.json()


In [206]:
print_json_structure(data)

features: list
  [list] len=17
    geometry: dict
      coordinates: list
        [list] len=2
      type: str
    properties: dict
      measurement: dict
        AQ_hourly_index: str
        components: list
          [list] len=2
            averaged_time: dict
              averaged_hours: str
              value: float
            type: str
      id: str
      name: str
      updated_at: str
      district: str
    type: str
type: str


In [207]:
df = pd.json_normalize(data['features'])

tmp = df.explode("properties.measurement.components", ignore_index=True)
dirs = pd.json_normalize(
    tmp["properties.measurement.components"]
).add_prefix("properties.measurement.components.")

df = pd.concat([tmp.drop(columns=["properties.measurement.components"]), dirs], axis=1)

df.head()

,type,geometry.coordinates,geometry.type,properties.measurement.AQ_hourly_index,properties.id,properties.name,properties.updated_at,properties.district,properties.measurement.components.type,properties.measurement.components.averaged_time.averaged_hours,properties.measurement.components.averaged_time.value
0,Feature,"[14.380116, 50.084385]",Point,2B,ABREA,Praha 6-Břevnov,2026-04-28T14:15:00.551Z,praha-6,NO2,3,9.8
1,Feature,"[14.380116, 50.084385]",Point,2B,ABREA,Praha 6-Břevnov,2026-04-28T14:15:00.551Z,praha-6,PM10,3,13.9
2,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T14:15:00.551Z,praha-7,NO2,3,16.4
3,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T14:15:00.551Z,praha-7,PM10,3,19.3
4,Feature,"[14.44365, 50.108845]",Point,2B,AHOLA,Praha 7-Holešovice,2026-04-28T14:15:00.551Z,praha-7,PM2_5,3,7.9


In [ ]:
#### air quality metadata processing

station_cols = {
    'geometry.coordinates': 'coordinates', 
    'properties.id': 'id', 
    'properties.name': 'name', 
    'properties.district': 'district', 
    'properties.measurement.components.type': 'components'
}

In [209]:
air_quality_stations = df[station_cols.keys()]

air_quality_stations = air_quality_stations.rename(columns=station_cols)

In [210]:
air_quality_stations.head()

,coordinates,id,name,district,components
0,"[14.380116, 50.084385]",ABREA,Praha 6-Břevnov,praha-6,NO2
1,"[14.380116, 50.084385]",ABREA,Praha 6-Břevnov,praha-6,PM10
2,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,NO2
3,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,PM10
4,"[14.44365, 50.108845]",AHOLA,Praha 7-Holešovice,praha-7,PM2_5


In [211]:
air_quality_stations = (
    air_quality_stations.groupby('id', as_index=False)
    .agg({
        'coordinates': 'first',
        'name': 'first',
        'district': 'first',
        'components': list
    })
)

In [216]:
air_quality_stations[["lon", "lat"]] = pd.DataFrame(air_quality_stations["coordinates"].tolist(), index=air_quality_stations.index)
air_quality_stations["lon"] = pd.to_numeric(air_quality_stations["lon"], errors="coerce")
air_quality_stations["lat"] = pd.to_numeric(air_quality_stations["lat"], errors="coerce")

In [217]:
type(air_quality_stations)

pandas.DataFrame

In [218]:
air_quality_stations.head()

,id,coordinates,name,district,components,lon,lat
0,ABREA,"[14.380116, 50.084385]",Praha 6-Břevnov,praha-6,"[NO2, PM10]",14.380116,50.084385
1,ACHOA,"[14.51745, 50.03017]",Praha 4-Chodov,praha-11,"[NO2, PM10]",14.517450,50.030170
2,AHOLA,"[14.44365, 50.108845]",Praha 7-Holešovice,praha-7,"[NO2, PM10, PM2_5]",14.443650,50.108845
3,AKALA,"[14.442049, 50.094238]",Praha 8-Karlín,praha-8,"[NO2, PM10]",14.442049,50.094238
4,AKOBA,"[14.467578, 50.122189]",Praha 8-Kobylisy,praha-8,"[NO2, O3, PM10]",14.467578,50.122189


In [ ]:
#air quality stations dictionary

air_stat_dict = dict(
    zip(
        air_quality_stations['id'].astype(str),
        air_quality_stations['name'].astype(str)  
    )
)

In [220]:
air_stat_dict

{'ABREA': 'Praha 6-Břevnov',
 'ACHOA': 'Praha 4-Chodov',
 'AHOLA': 'Praha 7-Holešovice',
 'AKALA': 'Praha 8-Karlín',
 'AKOBA': 'Praha 8-Kobylisy',
 'ALEGA': 'Praha 2-Legerova (hot spot)',
 'ALERA': 'Letiště Praha',
 'ALIBA': 'Praha 4-Libuš',
 'APRUA': 'Praha 10-Průmyslová',
 'AREPA': 'Praha 1-n. Republiky',
 'ARERA': 'Praha 5-Řeporyje',
 'ARIEA': 'Praha 2-Riegrovy sady',
 'ASROA': 'Praha 10-Šrobárova',
 'ASTOA': 'Praha 5-Stodůlky',
 'ASUCA': 'Praha 6-Suchdol',
 'AVRSA': 'Praha 10-Vršovice',
 'AVYNA': 'Praha 9-Vysočany'}

In [ ]:
### chmi weather data

# TODO to get temperature, we need 10min data 

In [ ]:
def get_chmi_weather_data(start_year = 2025, end_year = 2025):
    base_url = "https://opendata.chmi.cz/"
    route_template = "/meteorology/climate/historical/data/10min/{year}/10m-{wsi}-{ym}.json"

    url_header = {
        'accept': 'application/json',
        'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
    }

    years = [f"{y}" for y in range(start_year, end_year+1)]
    months = [f"{m:02d}" for m in range(1, 13)]

    results = []

    for wsi in wsi_dict:
        for year in years:
            for month in months:
                ym = f"{year}{month}"
                route = route_template.format(year=year, wsi=wsi, ym=ym)

                response = requests.get(
                    f"{base_url}{route}",
                    headers=url_header,
                    timeout=60
                )
                if response.status_code == 200:
                    data_response = response.json()
                    data_response = data_response['data']['data']

                    headers = data_response['header'].split(',')
                    values = data_response['values']

                    df_part = pd.DataFrame(values, columns=headers)
                    df_part["WSI"] = wsi
                    df_part["YEAR"] = year
                    df_part["MONTH"] = month

                    results.append(df_part)

                else:
                    print(f"Failed for {wsi} {wsi_dict[wsi]} {year} {month}: {response.status_code}")


    df_chmi = pd.concat(results, ignore_index=True)


In [171]:
base_url = "https://opendata.chmi.cz/"
route_template = "/meteorology/climate/historical/data/1hour/{year}/1h-{wsi}-{ym}.json"

url_header = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

years = [2025]
months = [11] #[f"{m:02d}" for m in range(1, 13)]

results = []

for wsi in wsi_dict:
    for year in years:
        for month in months:
            ym = f"{year}{month}"
            route = route_template.format(year=year, wsi=wsi, ym=ym)

            response = requests.get(
                f"{base_url}{route}",
                headers=url_header,
                timeout=60
            )
            if response.status_code == 200:
                data_response = response.json()
                data_response = data_response['data']['data']

                headers = data_response['header'].split(',')
                values = data_response['values']

                df_part = pd.DataFrame(values, columns=headers)
                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month

                results.append(df_part)

            else:
                print(f"Failed for {wsi} {wsi_dict[wsi]} {year} {month}: {response.status_code}")


df_chmi = pd.concat(results, ignore_index=True)


Failed for 0-203-0-11515 Praha, Klementinum 2025 11: 404
Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 11: 404
Failed for 0-203-0-11101007001 Praha, Brdy 2025 11: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 11: 404
Failed for 0-203-0-11201020003 Praha, Chodov 2025 11: 404


In [30]:
### for 10 mins

base_url = "https://opendata.chmi.cz/"
route_template = "/meteorology/climate/historical/data/10min/{year}/10m-{wsi}-{ym}.json"

url_header = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

years = [2025]
months = [11]

results = []

for wsi in wsi_dict:
    for year in years:
        for month in months:
            ym = f"{year}{month}"
            route = route_template.format(year=year, wsi=wsi, ym=ym)

            response = requests.get(
                f"{base_url}{route}",
                headers=url_header,
                timeout=60
            )
            if response.status_code == 200:
                data_response = response.json()
                data_response = data_response['data']['data']

                headers = data_response['header'].split(',')
                values = data_response['values']

                df_part = pd.DataFrame(values, columns=headers)
                df_part["WSI"] = wsi
                df_part["YEAR"] = year
                df_part["MONTH"] = month

                results.append(df_part)

            else:
                print(f"Failed for {wsi} {wsi_dict[wsi]} {year} {month}: {response.status_code}")


df_chmi = pd.concat(results, ignore_index=True)

Failed for 0-203-0-11105048001 Praha, Zadní Kopanina 2025 11: 404
Failed for 0-203-0-11202007001 Praha, Suchdol 2025 11: 404
Failed for 0-203-0-11201020003 Praha, Chodov 2025 11: 404


In [ ]:
### mapping variables names

df_chmi['ELEMENT_NAME'] = df_chmi['ELEMENT'].map(chmi_vars_dict)

NameError: name 'chmi_vars_dict' is not defined

In [33]:
df_chmi.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY,WSI,YEAR,MONTH
0,0-203-0-11201020001,Casmax,2025-11-01T00:00:00Z,568.0,,0.0,0-203-0-11201020001,2025,11
1,0-203-0-11201020001,Casmax,2025-11-01T00:10:00Z,547.0,,0.0,0-203-0-11201020001,2025,11
2,0-203-0-11201020001,Casmax,2025-11-01T00:20:00Z,4.0,,0.0,0-203-0-11201020001,2025,11
3,0-203-0-11201020001,Casmax,2025-11-01T00:30:00Z,1.0,,0.0,0-203-0-11201020001,2025,11
4,0-203-0-11201020001,Casmax,2025-11-01T00:40:00Z,315.0,,0.0,0-203-0-11201020001,2025,11


In [34]:
df_chmi.shape

(475164, 9)

In [36]:
df_chmi['ELEMENT'].unique()

<StringArray>
['Casmax',      'D',   'Dmax',  'Dprum',      'F',   'Fmax',  'Fprum',
      'H', 'SRA10M', 'SSV10M',      'T',    'TMA',    'TMI',    'TPM',
   'SCEa',      'P',    'T05',    'T10',   'T100',    'T20',    'T50',
 'RGLB10']
Length: 22, dtype: str

In [ ]:
### golemio 

### air quality data 

In [ ]:
api_url = 'https://api.golemio.cz/'
route = '/v2/airqualitystations/history'

headers = {
    'X-access-token': api_key, 
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

params = {
    'from': '2024-05-16T04:27:58.000Z', 
    'to': '2024-06-18T04:27:58.000Z'
}

response = requests.get(f'{api_url}{route}', headers=headers, params=params, timeout=60)

data = response.json()



In [243]:
print(json.dumps(data, indent=2))

{
  "error_message": "Not Found",
  "error_status": 404
}


In [232]:
# Debugging: Print all keys in the first measurement to find the hidden time
if data:
    print("Top level keys:", data[0].keys())
    print("Measurement level keys:", data[0]['measurement'].keys())

Top level keys: dict_keys(['id', 'measurement'])
Measurement level keys: dict_keys(['AQ_hourly_index', 'components'])


In [228]:
df = pd.json_normalize(
    data,
    record_path=["measurement", "components"],
    meta=[
        "id",
        ["measurement", "AQ_hourly_index"],
    ],
    sep=".",
    errors="raise",
)

In [229]:
df.head()

,type,averaged_time.averaged_hours,averaged_time.value,id,measurement.AQ_hourly_index
0,NO2,3,19.3,ABREA,1A
1,PM10,3,13.9,ABREA,1A
2,NO2,3,23.8,ABREA,1A
3,PM10,3,15.5,ABREA,1A
4,NO2,3,33.5,ABREA,1A


In [18]:
#### golemio

### microclimate sensors

In [19]:
# TODO if necessary